In [1]:
# Stage 2 Notebook: Data Expansion and Supervised Urgency Model

print("Stage 2 notebook initialised. This will handle:")
print("- Synthetic data expansion")
print("- Supervised urgency classification model")
print("- Evaluation vs the Stage 1 zero-shot baseline")


Stage 2 notebook initialised. This will handle:
- Synthetic data expansion
- Supervised urgency classification model
- Evaluation vs the Stage 1 zero-shot baseline


In [2]:
# Stage 2 – label sets for safeguarding classification

# Urgency levels (ordered from lowest to highest)
URGENCY_LABELS = ["Low", "Medium", "High", "Critical"]

URGENCY_LABEL_DESCRIPTIONS = {
    "Low": (
        "No immediate risk identified. Situation may require monitoring or signposting "
        "to support, but there is no indication of current harm or imminent danger."
    ),
    "Medium": (
        "Some safeguarding concern present. The young person may be impacted, but there "
        "is no indication of immediate serious harm. Requires timely follow-up."
    ),
    "High": (
        "Significant safeguarding concern. Indicators of harm or high vulnerability are "
        "present and the case requires prompt action and oversight."
    ),
    "Critical": (
        "Immediate or very serious risk of harm indicated. Requires urgent safeguarding "
        "response in line with organisational policy and escalation procedures."
    ),
}

# Concern categories (types of issues the system will recognise)
CATEGORY_LABELS = [
    "Bullying",
    "Physical safety",
    "Mental health",
    "Home issues",
    "Online safety",
    "Financial hardship",
    "Attendance / engagement",
    "Behaviour / conduct",
    "Other",
]

CATEGORY_LABEL_DESCRIPTIONS = {
    "Bullying": "Peer-on-peer bullying, harassment, exclusion or repeated unkind behaviour.",
    "Physical safety": "Injuries, physical abuse or situations where a young person may not be safe.",
    "Mental health": "Concerns about low mood, self-esteem, anxiety, self-harm or related issues.",
    "Home issues": "Family conflict, instability or home circumstances impacting the young person.",
    "Online safety": "Online bullying, grooming, unsafe contact or harmful online content.",
    "Financial hardship": "Difficulties affording transport, uniform, food or participation costs.",
    "Attendance / engagement": "Non-attendance, withdrawal from activities or sudden disengagement.",
    "Behaviour / conduct": "Behavioural changes, aggression, discipline issues or similar.",
    "Other": "Concerns that do not fit clearly into the other categories.",
}

print("Urgency labels:")
for u in URGENCY_LABELS:
    print(f" - {u}: {URGENCY_LABEL_DESCRIPTIONS[u]}")

print("\nConcern category labels:")
for c in CATEGORY_LABELS:
    print(f" - {c}: {CATEGORY_LABEL_DESCRIPTIONS[c]}")


Urgency labels:
 - Low: No immediate risk identified. Situation may require monitoring or signposting to support, but there is no indication of current harm or imminent danger.
 - Medium: Some safeguarding concern present. The young person may be impacted, but there is no indication of immediate serious harm. Requires timely follow-up.
 - High: Significant safeguarding concern. Indicators of harm or high vulnerability are present and the case requires prompt action and oversight.
 - Critical: Immediate or very serious risk of harm indicated. Requires urgent safeguarding response in line with organisational policy and escalation procedures.

Concern category labels:
 - Bullying: Peer-on-peer bullying, harassment, exclusion or repeated unkind behaviour.
 - Physical safety: Injuries, physical abuse or situations where a young person may not be safe.
 - Mental health: Concerns about low mood, self-esteem, anxiety, self-harm or related issues.
 - Home issues: Family conflict, instability or

In [3]:
import pandas as pd
import random
from datetime import datetime, timedelta

# We reuse URGENCY_LABELS and CATEGORY_LABELS from the previous cell.

# Helper lists for other fields
REPORTER_ROLES = ["Adult Volunteer", "Parent/Carer", "Cadet", "Other"]
LOCATIONS = [
    "Parade Night - Detachment",
    "Annual Camp",
    "Training Weekend",
    "Field Exercise",
    "Home",
]
AGE_BANDS = ["Under 12", "12-14", "15-17", "18+"]
CHANNELS = ["Online form", "Phone call", "Email", "In person", "Other"]
STATUS_VALUES = ["Open", "In progress", "Closed"]

# Simple unit IDs to sample from
UNIT_IDS = [f"NW-ACF-{str(i).zfill(3)}" for i in range(1, 51)]  # NW-ACF-001 ... NW-ACF-050

def generate_base_timestamp():
    """
    Generate a timestamp within a recent 90-day window.
    """
    now = datetime.now()
    days_back = random.randint(0, 90)
    random_time = now - timedelta(days=days_back, hours=random.randint(0, 23), minutes=random.randint(0, 59))
    return random_time.strftime("%Y-%m-%d %H:%M:%S")


def generate_free_text(category: str, urgency: str) -> str:
    """
    Generate a simple synthetic narrative based on category and urgency.
    This is deliberately templated but aims to resemble plausible safeguarding notes.
    """

    # Base stems by category
    category_stems = {
        "Bullying": (
            "Cadet has reported repeated unkind comments and exclusion from activities. "
            "They describe being targeted by a small group during recent events."
        ),
        "Physical safety": (
            "Cadet was observed with unexplained marks and appeared uncomfortable when asked. "
            "There are concerns about their physical safety outside of cadet activities."
        ),
        "Mental health": (
            "Cadet has become withdrawn and low in mood. They have mentioned struggling "
            "to cope and finding it hard to join in with normal activities."
        ),
        "Home issues": (
            "Cadet has hinted at ongoing arguments and tension at home. "
            "They say the atmosphere is difficult and it is affecting their behaviour."
        ),
        "Online safety": (
            "Cadet disclosed receiving unpleasant messages online and being added to "
            "group chats where inappropriate content is being shared."
        ),
        "Financial hardship": (
            "Cadet mentioned difficulties affording transport, uniform items and some "
            "optional activities, which may limit their ability to attend regularly."
        ),
        "Attendance / engagement": (
            "Cadet's attendance has dropped noticeably over recent weeks and they seem "
            "less engaged when they do attend."
        ),
        "Behaviour / conduct": (
            "Cadet has displayed changes in behaviour, including occasional outbursts "
            "and conflict with peers or staff."
        ),
        "Other": (
            "Cadet has raised a concern that does not fit neatly into another category, "
            "but staff feel it should be recorded and monitored."
        ),
    }

    # Urgency-specific modifiers
    urgency_modifiers = {
        "Low": (
            "At present there is no indication of immediate risk, but the situation "
            "will be monitored and support signposting considered."
        ),
        "Medium": (
            "The concern is impacting the cadet and requires timely follow-up, "
            "but there is no indication of immediate serious harm."
        ),
        "High": (
            "The concern suggests a significant safeguarding issue and the case "
            "requires prompt action and oversight from safeguarding leads."
        ),
        "Critical": (
            "There are indicators of immediate or very serious risk. Safeguarding "
            "procedures should be followed urgently in line with policy."
        ),
    }

    base = category_stems.get(category, "")
    mod = urgency_modifiers.get(urgency, "")
    return base + " " + mod


def generate_synthetic_report(report_id: int) -> dict:
    """
    Generate one synthetic safeguarding report record.
    """
    # Choose category and urgency
    category = random.choice(CATEGORY_LABELS)
    urgency = random.choice(URGENCY_LABELS)

    # Build the report dictionary
    report = {
        "report_id": report_id,
        "timestamp": generate_base_timestamp(),
        "unit_id": random.choice(UNIT_IDS),
        "reporter_role": random.choice(REPORTER_ROLES),
        "location": random.choice(LOCATIONS),
        "age_band": random.choice(AGE_BANDS),
        "channel": random.choice(CHANNELS),
        "free_text": generate_free_text(category, urgency),
        # Ground truth labels
        "category_label": category,
        "urgency_label": urgency,
        # Simple status assignment: more recent reports more likely Open/In progress
        "status": random.choice(STATUS_VALUES),
    }
    return report


# Number of synthetic reports to generate
NUM_SYNTHETIC_REPORTS = 200  # you can change this later if needed

synthetic_reports = [generate_synthetic_report(i + 1) for i in range(NUM_SYNTHETIC_REPORTS)]

synthetic_df = pd.DataFrame(synthetic_reports)

print("Preview of synthetic Stage 2 dataset:")
display(synthetic_df.head())

print("\nLabel distribution – urgency:")
print(synthetic_df["urgency_label"].value_counts())

print("\nLabel distribution – category:")
print(synthetic_df["category_label"].value_counts())

# Save to CSV in the processed data folder
output_path_stage2 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2.csv"
synthetic_df.to_csv(output_path_stage2, index=False)

print(f"\nSaved Stage 2 synthetic dataset to:\n{output_path_stage2}")


Preview of synthetic Stage 2 dataset:


,report_id,timestamp,unit_id,reporter_role,location,age_band,channel,free_text,category_label,urgency_label,status
0,1,2025-11-21 03:42:02,NW-ACF-021,Cadet,Training Weekend,18+,In person,Cadet disclosed receiving unpleasant messages ...,Online safety,Critical,Open
1,2,2025-12-17 10:49:02,NW-ACF-041,Cadet,Annual Camp,Under 12,Other,Cadet has hinted at ongoing arguments and tens...,Home issues,Critical,Closed
2,3,2026-01-14 16:22:02,NW-ACF-032,Adult Volunteer,Field Exercise,Under 12,Other,Cadet has raised a concern that does not fit n...,Other,Low,Open
3,4,2026-01-28 18:54:02,NW-ACF-032,Adult Volunteer,Parade Night - Detachment,18+,Email,Cadet has hinted at ongoing arguments and tens...,Home issues,Critical,Closed
4,5,2025-12-17 22:31:02,NW-ACF-012,Adult Volunteer,Home,12-14,Other,Cadet mentioned difficulties affording transpo...,Financial hardship,Critical,In progress



Label distribution – urgency:
urgency_label
High        55
Critical    54
Low         46
Medium      45
Name: count, dtype: int64

Label distribution – category:
category_label
Physical safety            30
Home issues                27
Bullying                   25
Financial hardship         25
Mental health              21
Attendance / engagement    20
Online safety              19
Other                      19
Behaviour / conduct        14
Name: count, dtype: int64

Saved Stage 2 synthetic dataset to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2.csv


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Path to the Stage 2 synthetic dataset created in Step 3
stage2_csv_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2.csv"

# 1. Load the dataset
df_stage2 = pd.read_csv(stage2_csv_path)

print("Loaded Stage 2 dataset.")
print("Number of reports:", len(df_stage2))
print("\nColumns:")
print(df_stage2.columns.tolist())

print("\nPreview of key fields:")
display(df_stage2[["report_id", "free_text", "category_label", "urgency_label"]].head())


# 2. Create a train/validation split for supervised modelling
#    We will predict urgency_label from free_text (and potentially other fields later).

train_df, val_df = train_test_split(
    df_stage2,
    test_size=0.2,         # 20% for validation
    random_state=42,       # fixed seed for reproducibility
    stratify=df_stage2["urgency_label"],  # keep urgency label distribution similar in both splits
)

print("\nTrain set size:", len(train_df))
print("Validation set size:", len(val_df))

print("\nUrgency label distribution in TRAIN set:")
print(train_df["urgency_label"].value_counts())

print("\nUrgency label distribution in VALIDATION set:")
print(val_df["urgency_label"].value_counts())


# 3. Save train and validation splits back to disk for later use
train_output_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2_train.csv"
val_output_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2_val.csv"

train_df.to_csv(train_output_path, index=False)
val_df.to_csv(val_output_path, index=False)

print("\nSaved train split to:")
print(train_output_path)

print("\nSaved validation split to:")
print(val_output_path)


Loaded Stage 2 dataset.
Number of reports: 200

Columns:
['report_id', 'timestamp', 'unit_id', 'reporter_role', 'location', 'age_band', 'channel', 'free_text', 'category_label', 'urgency_label', 'status']

Preview of key fields:


,report_id,free_text,category_label,urgency_label
0,1,Cadet disclosed receiving unpleasant messages ...,Online safety,Critical
1,2,Cadet has hinted at ongoing arguments and tens...,Home issues,Critical
2,3,Cadet has raised a concern that does not fit n...,Other,Low
3,4,Cadet has hinted at ongoing arguments and tens...,Home issues,Critical
4,5,Cadet mentioned difficulties affording transpo...,Financial hardship,Critical



Train set size: 160
Validation set size: 40

Urgency label distribution in TRAIN set:
urgency_label
High        44
Critical    43
Low         37
Medium      36
Name: count, dtype: int64

Urgency label distribution in VALIDATION set:
urgency_label
Critical    11
High        11
Medium       9
Low          9
Name: count, dtype: int64

Saved train split to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2_train.csv

Saved validation split to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2_val.csv


In [5]:
import torch
from transformers import AutoTokenizer

# We will use DistilBERT as the base model for supervised urgency classification
MODEL_NAME = "distilbert-base-uncased"

print("Using base model:", MODEL_NAME)

# Urgency labels (reuse the same order as earlier)
URGENCY_LABELS = ["Low", "Medium", "High", "Critical"]

# Create mappings between text labels and numeric IDs
label2id = {label: idx for idx, label in enumerate(URGENCY_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

print("\nLabel → ID mapping:")
for k, v in label2id.items():
    print(f"  {k} -> {v}")

print("\nID → Label mapping:")
for k, v in id2label.items():
    print(f"  {k} -> {v}")

# Load the tokenizer for the chosen model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nTokenizer loaded.")


Using base model: distilbert-base-uncased

Label → ID mapping:
  Low -> 0
  Medium -> 1
  High -> 2
  Critical -> 3

ID → Label mapping:
  0 -> Low
  1 -> Medium
  2 -> High
  3 -> Critical

Tokenizer loaded.


In [6]:
# Quick test: tokenize one training example from the Stage 2 train set

# Make sure we have train_df from earlier; if not, re-run the cell that created it
print("Train set size:", len(train_df))

example_row = train_df.iloc[0]
example_text = example_row["free_text"]
example_label_text = example_row["urgency_label"]
example_label_id = label2id[example_label_text]

print("\nExample free_text:")
print(example_text)
print("\nUrgency label (text):", example_label_text)
print("Urgency label (id):", example_label_id)

# Tokenise
encoded = tokenizer(
    example_text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt",  # return PyTorch tensors
)

print("\nTokenised example shapes:")
for k, v in encoded.items():
    print(f"  {k}: shape {tuple(v.shape)}")


Train set size: 160

Example free_text:
Cadet has raised a concern that does not fit neatly into another category, but staff feel it should be recorded and monitored. At present there is no indication of immediate risk, but the situation will be monitored and support signposting considered.

Urgency label (text): Low
Urgency label (id): 0

Tokenised example shapes:
  input_ids: shape (1, 128)
  attention_mask: shape (1, 128)


In [7]:
from torch.utils.data import Dataset

class SafeguardingUrgencyDataset(Dataset):
    """
    PyTorch Dataset for safeguarding urgency classification.

    - Expects a pandas DataFrame with at least:
        - 'free_text' (input text)
        - 'urgency_label' (text label, e.g. 'Low', 'Medium', 'High', 'Critical')
    - Uses the global tokenizer and label2id mapping defined earlier.
    """

    def __init__(self, dataframe, tokenizer, label2id, max_length: int = 128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["free_text"]
        label_text = row["urgency_label"]
        label_id = self.label2id[label_text]

        # Tokenise the text
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # encoded["input_ids"] etc. have shape (1, max_length); we want to remove the batch dim
        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(label_id, dtype=torch.long),
        }
        return item


# Create train and validation datasets
train_dataset = SafeguardingUrgencyDataset(train_df, tokenizer, label2id, max_length=128)
val_dataset = SafeguardingUrgencyDataset(val_df, tokenizer, label2id, max_length=128)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

# Quick sanity check on one example
example_item = train_dataset[0]
print("\nExample item keys:", list(example_item.keys()))
print("input_ids shape:", tuple(example_item["input_ids"].shape))
print("attention_mask shape:", tuple(example_item["attention_mask"].shape))
print("label id:", example_item["labels"].item(), "->", id2label[example_item["labels"].item()])


Train dataset size: 160
Validation dataset size: 40

Example item keys: ['input_ids', 'attention_mask', 'labels']
input_ids shape: (128,)
attention_mask shape: (128,)
label id: 0 -> Low


In [8]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

# Batch size for training/evaluation
BATCH_SIZE = 8

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train DataLoader batches:", len(train_dataloader))
print("Validation DataLoader batches:", len(val_dataloader))

# Set up device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

# Load DistilBERT model for sequence classification (urgency prediction)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(URGENCY_LABELS),
    id2label=id2label,
    label2id=label2id,
)

model.to(device)

print("Model loaded and moved to device.")


Train DataLoader batches: 20
Validation DataLoader batches: 5

Using device: cuda


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded and moved to device.


In [9]:
# Quick sanity check: run one batch through the model

# Get the first batch from the train DataLoader
batch = next(iter(train_dataloader))

# Move batch tensors to the same device as the model
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

print("Batch shapes:")
print("  input_ids:", tuple(input_ids.shape))
print("  attention_mask:", tuple(attention_mask.shape))
print("  labels:", tuple(labels.shape))

# Run a forward pass
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

loss = outputs.loss
logits = outputs.logits

print("\nForward pass successful.")
print("  Loss value:", float(loss.item()))
print("  Logits shape:", tuple(logits.shape))
print("  num_labels:", logits.shape[-1])


Batch shapes:
  input_ids: (8, 128)
  attention_mask: (8, 128)
  labels: (8,)

Forward pass successful.
  Loss value: 1.381584882736206
  Logits shape: (8, 4)
  num_labels: 4


In [10]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

# Training configuration
EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_dataloader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps,
)

print(f"Training for {EPOCHS} epochs with learning rate {LEARNING_RATE}.")


def evaluate_model(model, dataloader, device):
    """
    Simple evaluation: compute accuracy on the given dataloader.
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total if total > 0 else 0.0
    return accuracy


# Main training loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    print(f"\n=== Epoch {epoch + 1} / {EPOCHS} ===")

    for step, batch in enumerate(train_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

        # Print occasional progress
        if (step + 1) % 10 == 0 or (step + 1) == len(train_dataloader):
            avg_loss_so_far = running_loss / (step + 1)
            print(f"  Step {step + 1}/{len(train_dataloader)} - "
                  f"Avg train loss: {avg_loss_so_far:.4f}")

    # End of epoch: evaluate on validation set
    val_accuracy = evaluate_model(model, val_dataloader, device)
    avg_train_loss = running_loss / len(train_dataloader)

    print(f"\nEpoch {epoch + 1} summary:")
    print(f"  Average training loss: {avg_train_loss:.4f}")
    print(f"  Validation accuracy:   {val_accuracy:.4f}")


Training for 3 epochs with learning rate 2e-05.

=== Epoch 1 / 3 ===
  Step 10/20 - Avg train loss: 1.3352
  Step 20/20 - Avg train loss: 1.2359

Epoch 1 summary:
  Average training loss: 1.2359
  Validation accuracy:   1.0000

=== Epoch 2 / 3 ===
  Step 10/20 - Avg train loss: 0.8331
  Step 20/20 - Avg train loss: 0.7082

Epoch 2 summary:
  Average training loss: 0.7082
  Validation accuracy:   1.0000

=== Epoch 3 / 3 ===
  Step 10/20 - Avg train loss: 0.4582
  Step 20/20 - Avg train loss: 0.4252

Epoch 3 summary:
  Average training loss: 0.4252
  Validation accuracy:   1.0000


In [11]:
import os

# Directory to save the fine-tuned urgency model
model_save_dir = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\urgency_distilbert_stage2"

os.makedirs(model_save_dir, exist_ok=True)

print("Saving model and tokenizer to:")
print(model_save_dir)

# Save the fine-tuned model
model.save_pretrained(model_save_dir)

# Save the tokenizer (so we use the exact same tokenisation at inference time)
tokenizer.save_pretrained(model_save_dir)

print("Save complete.")


Saving model and tokenizer to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\urgency_distilbert_stage2
Save complete.


In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# Reload model and tokenizer from disk
print("Reloading model and tokenizer from disk...")

loaded_tokenizer = AutoTokenizer.from_pretrained(model_save_dir)
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    model_save_dir,
    id2label=id2label,
    label2id=label2id,
)

loaded_model.to(device)
loaded_model.eval()

print("Reload complete. Model is on device:", device)


def predict_urgency(text: str) -> dict:
    """
    Use the fine-tuned DistilBERT model to predict urgency for a single text.

    Returns:
      - predicted_label: string urgency label (e.g. 'High')
      - probs: dict of {label: probability}
    """
    # Tokenise
    encoded = loaded_tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = loaded_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits = outputs.logits
        probs_tensor = F.softmax(logits, dim=-1).squeeze(0)

    probs = probs_tensor.cpu().tolist()
    label_probs = {id2label[i]: float(probs[i]) for i in range(len(probs))}

    # Pick the best label
    predicted_id = int(torch.argmax(probs_tensor).item())
    predicted_label = id2label[predicted_id]

    return {
        "predicted_label": predicted_label,
        "probs": label_probs,
    }


# Quick test on a sample text
test_text = (
    "Cadet arrived very quiet and tearful. When spoken to privately, they hinted "
    "that someone at home loses their temper and they feel scared about going back."
)

result = predict_urgency(test_text)

print("\nTest text:")
print(test_text)
print("\nPredicted urgency:", result["predicted_label"])
print("Probabilities:")
for label, p in result["probs"].items():
    print(f"  {label}: {p:.3f}")


Reloading model and tokenizer from disk...
Reload complete. Model is on device: cuda

Test text:
Cadet arrived very quiet and tearful. When spoken to privately, they hinted that someone at home loses their temper and they feel scared about going back.

Predicted urgency: Low
Probabilities:
  Low: 0.320
  Medium: 0.242
  High: 0.247
  Critical: 0.191


In [13]:
import pandas as pd

# Hand-crafted evaluation examples.
# These are NOT used for training – they simulate realistic narratives
# with human-assigned "intended" urgency labels.

manual_examples = [
    {
        "example_id": 1,
        "intended_urgency": "Low",
        "text": (
            "Cadet has been slightly quieter than usual and missed one activity due to "
            "schoolwork. When asked, they said they feel fine but are tired with exams."
        ),
    },
    {
        "example_id": 2,
        "intended_urgency": "Medium",
        "text": (
            "Cadet has started sitting on their own and avoiding group activities. "
            "They said they feel stressed and down but denied any thoughts of harming themselves."
        ),
    },
    {
        "example_id": 3,
        "intended_urgency": "High",
        "text": (
            "Cadet disclosed that they have been struggling to eat and sleep properly and "
            "have thought about hurting themselves, but did not mention a specific plan."
        ),
    },
    {
        "example_id": 4,
        "intended_urgency": "Critical",
        "text": (
            "Cadet stated that they have a plan to harm themselves tonight and described "
            "how they intend to do it. They appeared extremely distressed during the conversation."
        ),
    },
    {
        "example_id": 5,
        "intended_urgency": "High",
        "text": (
            "Cadet arrived with noticeable bruises on their arms and became very anxious when "
            "asked how they occurred. They hinted that someone at home loses their temper a lot."
        ),
    },
    {
        "example_id": 6,
        "intended_urgency": "Medium",
        "text": (
            "Parent contacted the unit to say there have been frequent arguments at home and "
            "the cadet is finding it hard to concentrate, but there has been no physical violence."
        ),
    },
    {
        "example_id": 7,
        "intended_urgency": "Low",
        "text": (
            "Cadet mentioned that they may need help with transport costs to attend more "
            "activities, but otherwise seemed positive and engaged."
        ),
    },
    {
        "example_id": 8,
        "intended_urgency": "Critical",
        "text": (
            "Another cadet reported seeing threatening messages sent to this cadet online, "
            "including statements that they will be hurt after the next parade night."
        ),
    },
]

manual_df = pd.DataFrame(manual_examples)

print("Manual evaluation examples:")
display(manual_df[["example_id", "intended_urgency", "text"]])


Manual evaluation examples:


,example_id,intended_urgency,text
0,1,Low,Cadet has been slightly quieter than usual and...
1,2,Medium,Cadet has started sitting on their own and avo...
2,3,High,Cadet disclosed that they have been struggling...
3,4,Critical,Cadet stated that they have a plan to harm the...
4,5,High,Cadet arrived with noticeable bruises on their...
5,6,Medium,Parent contacted the unit to say there have be...
6,7,Low,Cadet mentioned that they may need help with t...
7,8,Critical,Another cadet reported seeing threatening mess...


In [14]:
# Use the fine-tuned model (predict_urgency function) on the manual examples

results = []

for _, row in manual_df.iterrows():
    text = row["text"]
    intended = row["intended_urgency"]

    pred = predict_urgency(text)
    predicted_label = pred["predicted_label"]
    probs = pred["probs"]

    results.append({
        "example_id": row["example_id"],
        "intended_urgency": intended,
        "predicted_urgency": predicted_label,
        "Low_prob": probs["Low"],
        "Medium_prob": probs["Medium"],
        "High_prob": probs["High"],
        "Critical_prob": probs["Critical"],
    })

manual_eval_df = pd.DataFrame(results)

print("Manual evaluation – model vs intended labels:\n")
display(manual_eval_df[["example_id", "intended_urgency", "predicted_urgency"]])

# Simple accuracy on this manual set
correct = (manual_eval_df["intended_urgency"] == manual_eval_df["predicted_urgency"]).sum()
total = len(manual_eval_df)
accuracy = correct / total if total > 0 else 0.0

print(f"\nManual set accuracy: {correct} / {total} = {accuracy:.2f}")


Manual evaluation – model vs intended labels:



,example_id,intended_urgency,predicted_urgency
0,1,Low,Low
1,2,Medium,Low
2,3,High,Low
3,4,Critical,Medium
4,5,High,High
5,6,Medium,Medium
6,7,Low,Low
7,8,Critical,Medium



Manual set accuracy: 4 / 8 = 0.50


In [15]:
manual_eval_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\reports\manual_urgency_evaluation_stage2.csv"
manual_eval_df.to_csv(manual_eval_path, index=False)

print("Saved manual evaluation results to:")
print(manual_eval_path)


Saved manual evaluation results to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\reports\manual_urgency_evaluation_stage2.csv


In [16]:
import random

# Stage 2b – richer synthetic data with more varied narratives and high-risk scenarios

# Expanded category labels for Stage 2b
CATEGORY_LABELS_V2 = [
    "Bullying",
    "Physical safety",
    "Mental health",
    "Home issues",
    "Online safety",
    "Financial hardship",
    "Attendance / engagement",
    "Behaviour / conduct",
    "Abuse by adult in organisation",
    "Radicalisation / extremism",
    "Exploitation / trafficking",
    "FGM / harmful practices",
    "Other",
]

# High-risk categories where we expect urgency to skew towards High/Critical
HIGH_RISK_CATEGORIES = {
    "Abuse by adult in organisation",
    "Radicalisation / extremism",
    "Exploitation / trafficking",
    "FGM / harmful practices",
    "Physical safety",
    "Serious self-harm / suicidality",  # used in text but not as a separate label
}

# Multiple stems per category (richer narratives)
CATEGORY_STEMS_V2 = {
    "Bullying": [
        "Cadet reports repeated unkind comments and name-calling from a small group during parade nights.",
        "Several peers have been excluding the cadet from group activities and making jokes at their expense.",
        "Cadet disclosed that messages in the detachment group chat have become hostile and targeted towards them.",
    ],
    "Physical safety": [
        "Cadet attended with visible bruising and flinched when staff moved nearby, becoming distressed when asked about the injuries.",
        "Cadet mentioned being pushed or grabbed at home and appeared anxious about returning there after activities.",
        "Staff observed the cadet struggling with mobility and wincing in pain during physical tasks, but they were reluctant to explain why.",
    ],
    "Mental health": [
        "Cadet has appeared increasingly withdrawn and low in mood over several weeks, participating less in activities they previously enjoyed.",
        "Cadet described feeling constantly stressed and overwhelmed and said they are finding it hard to cope with day-to-day tasks.",
        "Cadet has spoken about feeling ‘numb’ and disconnected from friends and activities, and has stopped engaging in usual hobbies.",
    ],
    "Home issues": [
        "Cadet described frequent shouting and arguments at home, saying the atmosphere is tense and they struggle to relax there.",
        "Cadet hinted that there are ongoing problems between carers at home and that they often stay out late to avoid being there.",
        "Cadet reported that there have been changes in who is living at home and that they do not feel comfortable around one of the adults.",
    ],
    "Online safety": [
        "Cadet showed staff messages received on social media containing threats and humiliating comments from peers.",
        "Cadet reported being added to group chats where explicit or disturbing content is shared, and feels pressured to remain in them.",
        "Cadet disclosed that an older individual they met online has been asking for personal information and to meet in person.",
    ],
    "Financial hardship": [
        "Cadet explained that they are struggling to afford transport to activities and have missed several parades as a result.",
        "Parent mentioned that uniform and boot costs are a concern and that finances are affecting the cadet’s ability to attend events.",
        "Cadet quietly asked if there was any support available with food during longer activities, as things are difficult at home.",
    ],
    "Attendance / engagement": [
        "Cadet’s attendance has dropped sharply over the last month and they appear distracted when they do attend.",
        "Cadet, who was previously enthusiastic, has begun arriving late, leaving early and avoiding taking part in team tasks.",
        "Staff have noticed the cadet regularly missing key training sessions without clear reasons, and peers have expressed concern.",
    ],
    "Behaviour / conduct": [
        "Cadet has become increasingly argumentative with peers and staff, with incidents of raised voices and refusal to follow instructions.",
        "There have been a number of low-level altercations involving the cadet, including pushing and verbal disputes.",
        "Cadet has been observed winding up younger cadets and ignoring reminders about behaviour standards.",
    ],
    "Abuse by adult in organisation": [
        "Cadet disclosed that a Cadet Force Adult Volunteer (CFAV) has made them feel uncomfortable with comments and unnecessary physical contact during activities.",
        "Another cadet reported seeing a CFAV regularly isolate this cadet from the group and speak to them behind closed doors.",
        "Cadet stated that a CFAV has been sending them personal messages outside normal communication channels and has hinted at meeting one-to-one.",
    ],
    "Radicalisation / extremism": [
        "Cadet has been talking positively about extremist groups and has shared online content that appears to support violent ideologies.",
        "Cadet mentioned being in private online forums where extremist views are discussed and has repeated some of these views at parade.",
        "Another cadet reported that this cadet has spoken about wanting to travel to conflict zones and ‘fight for a cause’.",
    ],
    "Exploitation / trafficking": [
        "Cadet disclosed that an older individual outside the organisation regularly collects them late at night and provides gifts in exchange for ‘favours’.",
        "Staff were told that the cadet has been missing from home overnight on several occasions and returns with unexplained money or items.",
        "Cadet hinted that they are being pressured to go to addresses with people they do not know well and feel unable to say no.",
    ],
    "FGM / harmful practices": [
        "Cadet disclosed that relatives have spoken about a ‘special procedure’ to be carried out when they visit family abroad and appeared very anxious about this.",
        "Parent conversation suggested that the family is planning a trip where cultural practices may include procedures that pose a risk of serious harm.",
        "Cadet asked general questions about whether certain traditional practices on girls are allowed in this country and appeared worried.",
    ],
    "Other": [
        "Cadet raised a concern that does not fit a specific category but left staff feeling that it should be recorded and monitored.",
        "During a general discussion, cadet disclosed a situation that may indicate vulnerability, though details are not yet clear.",
        "Third-party information was received suggesting there may be welfare concerns for this cadet, but further clarification is needed.",
    ],
}

# Urgency-specific modifiers (multiple variants)
URGENCY_MODIFIERS_V2 = {
    "Low": [
        "At present there is no indication of immediate risk, but the situation will be monitored and support options explained to the cadet.",
        "The concern appears low-level at this stage; staff will continue to observe and provide a safe space for the cadet to talk.",
        "No urgent safeguarding action is required currently, though the information has been recorded and will be reviewed periodically.",
    ],
    "Medium": [
        "The concern is having an impact on the cadet and requires timely follow-up in line with safeguarding procedures.",
        "Staff have agreed to monitor the situation closely and to share information with the safeguarding lead for further review.",
        "Further conversation with the cadet and parents/carers is needed to understand the situation and decide on next steps.",
    ],
    "High": [
        "The information suggests a significant safeguarding concern and the case requires prompt discussion with the safeguarding lead.",
        "Safeguarding procedures should be followed without delay, with a clear record of actions and decisions taken.",
        "The case will be escalated to the appropriate safeguarding officer and external agencies may need to be consulted.",
    ],
    "Critical": [
        "There are indicators of immediate or very serious risk and urgent safeguarding action is required in line with policy.",
        "The safeguarding lead must be informed immediately and emergency procedures may need to be initiated.",
        "Contact with external safeguarding agencies and, if necessary, emergency services should be considered without delay.",
    ],
}


def generate_free_text_v2(category: str, urgency: str) -> str:
    """
    Generate a richer synthetic narrative for Stage 2b based on category and urgency.

    - Picks a random stem from CATEGORY_STEMS_V2 for the given category.
    - Picks a random urgency modifier from URGENCY_MODIFIERS_V2 for the given urgency.
    - Occasionally adds an additional neutral clause for variety.
    """

    # Pick a category stem
    stems = CATEGORY_STEMS_V2.get(category, CATEGORY_STEMS_V2["Other"])
    base_stem = random.choice(stems)

    # Pick an urgency modifier
    mods = URGENCY_MODIFIERS_V2.get(urgency, URGENCY_MODIFIERS_V2["Medium"])
    mod = random.choice(mods)

    # Optional neutral clause to add variety
    neutral_clauses = [
        "The cadet has attended most recent parades but staff have noticed changes in presentation and demeanour.",
        "Peers have quietly expressed concern about the cadet and asked staff to check whether they are okay.",
        "The information has been logged on the safeguarding system and will be reviewed at the next supervision meeting.",
        "Staff are considering whether any immediate practical support can be offered to reduce pressure on the cadet.",
    ]
    add_neutral = random.random() < 0.5  # 50% chance

    text = base_stem + " " + mod
    if add_neutral:
        text = text + " " + random.choice(neutral_clauses)

    return text


In [17]:
import pandas as pd
from datetime import datetime, timedelta

# We reuse:
# - CATEGORY_LABELS_V2
# - CATEGORY_STEMS_V2
# - URGENCY_MODIFIERS_V2
# - generate_free_text_v2
# from Stage 2b Step 1

# Urgency labels (same as before)
URGENCY_LABELS = ["Low", "Medium", "High", "Critical"]

# Reporter roles – include CFAV explicitly
REPORTER_ROLES_V2 = [
    "Adult Volunteer (CFAV)",
    "Parent/Carer",
    "Cadet",
    "Other",
]

LOCATIONS_V2 = [
    "Parade Night - Detachment",
    "Annual Camp",
    "Training Weekend",
    "Field Exercise",
    "Home",
    "Online session",
]

AGE_BANDS_V2 = ["Under 12", "12-14", "15-17", "18+"]

CHANNELS_V2 = ["Online form", "Phone call", "Email", "In person", "Other"]

STATUS_VALUES_V2 = ["Open", "In progress", "Closed"]

UNIT_IDS_V2 = [f"NW-ACF-{str(i).zfill(3)}" for i in range(1, 101)]  # NW-ACF-001 ... NW-ACF-100

# Category sampling weights – we bias towards some high-risk categories so they appear more often
CATEGORY_SAMPLING_WEIGHTS = {
    "Bullying": 1.0,
    "Physical safety": 1.3,
    "Mental health": 1.2,
    "Home issues": 1.2,
    "Online safety": 1.0,
    "Financial hardship": 0.8,
    "Attendance / engagement": 0.8,
    "Behaviour / conduct": 0.8,
    "Abuse by adult in organisation": 1.5,
    "Radicalisation / extremism": 1.3,
    "Exploitation / trafficking": 1.3,
    "FGM / harmful practices": 1.3,
    "Other": 0.8,
}

def sample_category_v2():
    """
    Sample a category from CATEGORY_LABELS_V2 using CATEGORY_SAMPLING_WEIGHTS.
    """
    weights = [CATEGORY_SAMPLING_WEIGHTS.get(cat, 1.0) for cat in CATEGORY_LABELS_V2]
    return random.choices(CATEGORY_LABELS_V2, weights=weights, k=1)[0]


def sample_urgency_for_category_v2(category: str) -> str:
    """
    Sample an urgency label given a category.

    - For high-risk categories (e.g. CFAV abuse, radicalisation, trafficking, FGM, physical safety),
      we skew towards High and Critical.
    - For other categories, we skew towards Low and Medium, with some High/Critical.
    """

    high_risk_categories = {
        "Abuse by adult in organisation",
        "Radicalisation / extremism",
        "Exploitation / trafficking",
        "FGM / harmful practices",
        "Physical safety",
    }

    if category in high_risk_categories:
        # High-risk: more High/Critical
        # Probabilities roughly: Low 0.05, Medium 0.25, High 0.35, Critical 0.35
        choices = URGENCY_LABELS
        weights = [0.05, 0.25, 0.35, 0.35]
        return random.choices(choices, weights=weights, k=1)[0]
    else:
        # Other categories: more Low/Medium
        # Probabilities roughly: Low 0.30, Medium 0.40, High 0.20, Critical 0.10
        choices = URGENCY_LABELS
        weights = [0.30, 0.40, 0.20, 0.10]
        return random.choices(choices, weights=weights, k=1)[0]


def generate_base_timestamp_v2():
    """
    Generate a timestamp within a recent 180-day window.
    """
    now = datetime.now()
    days_back = random.randint(0, 180)
    random_time = now - timedelta(days=days_back, hours=random.randint(0, 23), minutes=random.randint(0, 59))
    return random_time.strftime("%Y-%m-%d %H:%M:%S")


def generate_synthetic_report_v2(report_id: int) -> dict:
    """
    Generate one synthetic safeguarding report record for Stage 2b
    using the richer narrative generator and risk-aware urgency sampling.
    """

    # 1. Choose category and urgency
    category = sample_category_v2()
    urgency = sample_urgency_for_category_v2(category)

    # 2. Generate narrative text
    free_text = generate_free_text_v2(category, urgency)

    # 3. Reporter role – if category is 'Abuse by adult in organisation', there is a higher chance
    #    the concern is reported by a cadet or a peer/parent rather than the CFAV themselves.
    if category == "Abuse by adult in organisation":
        reporter_role = random.choices(
            ["Cadet", "Parent/Carer", "Adult Volunteer (CFAV)", "Other"],
            weights=[0.5, 0.3, 0.1, 0.1],
            k=1,
        )[0]
    else:
        reporter_role = random.choice(REPORTER_ROLES_V2)

    report = {
        "report_id": report_id,
        "timestamp": generate_base_timestamp_v2(),
        "unit_id": random.choice(UNIT_IDS_V2),
        "reporter_role": reporter_role,
        "location": random.choice(LOCATIONS_V2),
        "age_band": random.choice(AGE_BANDS_V2),
        "channel": random.choice(CHANNELS_V2),
        "free_text": free_text,
        # Ground truth labels
        "category_label": category,
        "urgency_label": urgency,
        # Status
        "status": random.choice(STATUS_VALUES_V2),
    }

    return report


# Number of synthetic reports for Stage 2b – can adjust if needed
NUM_SYNTHETIC_REPORTS_V2 = 1000

synthetic_reports_v2 = [generate_synthetic_report_v2(i + 1) for i in range(NUM_SYNTHETIC_REPORTS_V2)]

synthetic_df_v2 = pd.DataFrame(synthetic_reports_v2)

print("Preview of Stage 2b synthetic dataset:")
display(synthetic_df_v2.head())

print("\nLabel distribution – urgency:")
print(synthetic_df_v2["urgency_label"].value_counts())

print("\nLabel distribution – category:")
print(synthetic_df_v2["category_label"].value_counts())


# Save to CSV in the processed data folder (new file, do NOT overwrite stage 2!)
output_path_stage2b = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b.csv"
synthetic_df_v2.to_csv(output_path_stage2b, index=False)

print(f"\nSaved Stage 2b synthetic dataset to:\n{output_path_stage2b}")


Preview of Stage 2b synthetic dataset:


,report_id,timestamp,unit_id,reporter_role,location,age_band,channel,free_text,category_label,urgency_label,status
0,1,2026-01-17 17:52:08,NW-ACF-030,Cadet,Field Exercise,Under 12,Phone call,Cadet asked general questions about whether ce...,FGM / harmful practices,Medium,Open
1,2,2025-12-19 00:49:08,NW-ACF-052,Parent/Carer,Training Weekend,Under 12,Online form,Cadet disclosed that a Cadet Force Adult Volun...,Abuse by adult in organisation,Low,In progress
2,3,2026-01-22 22:33:08,NW-ACF-079,Other,Annual Camp,Under 12,Online form,Another cadet reported seeing a CFAV regularly...,Abuse by adult in organisation,High,Closed
3,4,2026-01-09 19:22:08,NW-ACF-002,Other,Field Exercise,15-17,In person,Cadet attended with visible bruising and flinc...,Physical safety,Medium,In progress
4,5,2025-10-31 18:26:08,NW-ACF-004,Other,Home,12-14,Online form,Cadet has become increasingly argumentative wi...,Behaviour / conduct,Medium,Open



Label distribution – urgency:
urgency_label
Medium      340
High        261
Critical    217
Low         182
Name: count, dtype: int64

Label distribution – category:
category_label
Abuse by adult in organisation    112
Physical safety                   106
Exploitation / trafficking         95
Home issues                        88
Radicalisation / extremism         88
Mental health                      81
Online safety                      71
Behaviour / conduct                69
FGM / harmful practices            68
Other                              63
Bullying                           61
Attendance / engagement            50
Financial hardship                 48
Name: count, dtype: int64

Saved Stage 2b synthetic dataset to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b.csv


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Path to the richer Stage 2b synthetic dataset
stage2b_csv_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b.csv"

# 1. Load the Stage 2b dataset
df_stage2b = pd.read_csv(stage2b_csv_path)

print("Loaded Stage 2b dataset.")
print("Number of reports:", len(df_stage2b))
print("\nColumns:")
print(df_stage2b.columns.tolist())

print("\nPreview of key fields:")
display(df_stage2b[["report_id", "category_label", "urgency_label", "reporter_role", "free_text"]].head())

# 2. Create a train/validation split for supervised modelling (urgency prediction)
train_df_v2, val_df_v2 = train_test_split(
    df_stage2b,
    test_size=0.2,                        # 20% for validation
    random_state=42,                      # fixed seed for reproducibility
    stratify=df_stage2b["urgency_label"], # keep urgency label distribution similar in both splits
)

print("\nStage 2b train/validation sizes:")
print("  Train set size:", len(train_df_v2))
print("  Validation set size:", len(val_df_v2))

print("\nUrgency label distribution in TRAIN set (Stage 2b):")
print(train_df_v2["urgency_label"].value_counts())

print("\nUrgency label distribution in VALIDATION set (Stage 2b):")
print(val_df_v2["urgency_label"].value_counts())

# 3. Save Stage 2b train/validation splits
train_output_path_v2 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b_train.csv"
val_output_path_v2 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b_val.csv"

train_df_v2.to_csv(train_output_path_v2, index=False)
val_df_v2.to_csv(val_output_path_v2, index=False)

print("\nSaved Stage 2b train split to:")
print(train_output_path_v2)

print("\nSaved Stage 2b validation split to:")
print(val_output_path_v2)


Loaded Stage 2b dataset.
Number of reports: 1000

Columns:
['report_id', 'timestamp', 'unit_id', 'reporter_role', 'location', 'age_band', 'channel', 'free_text', 'category_label', 'urgency_label', 'status']

Preview of key fields:


,report_id,category_label,urgency_label,reporter_role,free_text
0,1,FGM / harmful practices,Medium,Cadet,Cadet asked general questions about whether ce...
1,2,Abuse by adult in organisation,Low,Parent/Carer,Cadet disclosed that a Cadet Force Adult Volun...
2,3,Abuse by adult in organisation,High,Other,Another cadet reported seeing a CFAV regularly...
3,4,Physical safety,Medium,Other,Cadet attended with visible bruising and flinc...
4,5,Behaviour / conduct,Medium,Other,Cadet has become increasingly argumentative wi...



Stage 2b train/validation sizes:
  Train set size: 800
  Validation set size: 200

Urgency label distribution in TRAIN set (Stage 2b):
urgency_label
Medium      272
High        209
Critical    173
Low         146
Name: count, dtype: int64

Urgency label distribution in VALIDATION set (Stage 2b):
urgency_label
Medium      68
High        52
Critical    44
Low         36
Name: count, dtype: int64

Saved Stage 2b train split to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b_train.csv

Saved Stage 2b validation split to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b_val.csv


In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

# Stage 2b – fresh DistilBERT model and data loaders

# Urgency labels – same as before
URGENCY_LABELS = ["Low", "Medium", "High", "Critical"]
label2id_v2 = {label: idx for idx, label in enumerate(URGENCY_LABELS)}
id2label_v2 = {idx: label for label, idx in label2id_v2.items()}

print("Stage 2b label → ID mapping:")
for k, v in label2id_v2.items():
    print(f"  {k} -> {v}")

# Tokenizer (same base model)
MODEL_NAME = "distilbert-base-uncased"
tokenizer_v2 = AutoTokenizer.from_pretrained(MODEL_NAME)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

# Reuse the SafeguardingUrgencyDataset class defined earlier
train_dataset_v2 = SafeguardingUrgencyDataset(train_df_v2, tokenizer_v2, label2id_v2, max_length=128)
val_dataset_v2 = SafeguardingUrgencyDataset(val_df_v2, tokenizer_v2, label2id_v2, max_length=128)

print("\nStage 2b dataset sizes:")
print("  Train dataset size:", len(train_dataset_v2))
print("  Validation dataset size:", len(val_dataset_v2))

# DataLoaders
BATCH_SIZE_V2 = 8
train_dataloader_v2 = DataLoader(train_dataset_v2, batch_size=BATCH_SIZE_V2, shuffle=True)
val_dataloader_v2 = DataLoader(val_dataset_v2, batch_size=BATCH_SIZE_V2, shuffle=False)

print("\nStage 2b DataLoader batches:")
print("  Train batches:", len(train_dataloader_v2))
print("  Validation batches:", len(val_dataloader_v2))

# Fresh DistilBERT model for sequence classification
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(URGENCY_LABELS),
    id2label=id2label_v2,
    label2id=label2id_v2,
)

model_v2.to(device)

print("\nStage 2b model loaded and moved to device.")


Stage 2b label → ID mapping:
  Low -> 0
  Medium -> 1
  High -> 2
  Critical -> 3

Using device: cuda

Stage 2b dataset sizes:
  Train dataset size: 800
  Validation dataset size: 200

Stage 2b DataLoader batches:
  Train batches: 100
  Validation batches: 25


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Stage 2b model loaded and moved to device.


In [20]:
from transformers import get_linear_schedule_with_warmup

# Training configuration for Stage 2b
EPOCHS_V2 = 3
LEARNING_RATE_V2 = 2e-5

optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LEARNING_RATE_V2)

total_steps_v2 = len(train_dataloader_v2) * EPOCHS_V2

scheduler_v2 = get_linear_schedule_with_warmup(
    optimizer_v2,
    num_warmup_steps=0,
    num_training_steps=total_steps_v2,
)

print(f"Stage 2b – training for {EPOCHS_V2} epochs with learning rate {LEARNING_RATE_V2}.")


def evaluate_model_v2(model, dataloader, device):
    """
    Simple accuracy evaluation for Stage 2b.
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total if total > 0 else 0.0
    return accuracy


# Main training loop for Stage 2b
for epoch in range(EPOCHS_V2):
    model_v2.train()
    running_loss = 0.0

    print(f"\n=== Stage 2b – Epoch {epoch + 1} / {EPOCHS_V2} ===")

    for step, batch in enumerate(train_dataloader_v2):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_v2.zero_grad()

        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), max_norm=1.0)

        optimizer_v2.step()
        scheduler_v2.step()

        running_loss += loss.item()

        if (step + 1) % 20 == 0 or (step + 1) == len(train_dataloader_v2):
            avg_loss_so_far = running_loss / (step + 1)
            print(f"  Step {step + 1}/{len(train_dataloader_v2)} - "
                  f"Avg train loss: {avg_loss_so_far:.4f}")

    # End of epoch – evaluate on validation set
    val_accuracy_v2 = evaluate_model_v2(model_v2, val_dataloader_v2, device)
    avg_train_loss_v2 = running_loss / len(train_dataloader_v2)

    print(f"\nStage 2b – Epoch {epoch + 1} summary:")
    print(f"  Average training loss: {avg_train_loss_v2:.4f}")
    print(f"  Validation accuracy:   {val_accuracy_v2:.4f}")


Stage 2b – training for 3 epochs with learning rate 2e-05.

=== Stage 2b – Epoch 1 / 3 ===
  Step 20/100 - Avg train loss: 1.3775
  Step 40/100 - Avg train loss: 1.3176
  Step 60/100 - Avg train loss: 1.1678
  Step 80/100 - Avg train loss: 0.9695
  Step 100/100 - Avg train loss: 0.8052

Stage 2b – Epoch 1 summary:
  Average training loss: 0.8052
  Validation accuracy:   1.0000

=== Stage 2b – Epoch 2 / 3 ===
  Step 20/100 - Avg train loss: 0.0653
  Step 40/100 - Avg train loss: 0.0529
  Step 60/100 - Avg train loss: 0.0454
  Step 80/100 - Avg train loss: 0.0403
  Step 100/100 - Avg train loss: 0.0365

Stage 2b – Epoch 2 summary:
  Average training loss: 0.0365
  Validation accuracy:   1.0000

=== Stage 2b – Epoch 3 / 3 ===
  Step 20/100 - Avg train loss: 0.0192
  Step 40/100 - Avg train loss: 0.0183
  Step 60/100 - Avg train loss: 0.0179
  Step 80/100 - Avg train loss: 0.0173
  Step 100/100 - Avg train loss: 0.0170

Stage 2b – Epoch 3 summary:
  Average training loss: 0.0170
  Validati

In [21]:
import torch.nn.functional as F

# Stage 2b – prediction helper using the new model_v2

def predict_urgency_v2(text: str) -> dict:
    """
    Use the Stage 2b fine-tuned DistilBERT model (model_v2) to predict urgency for a single text.

    Returns:
      - predicted_label: string urgency label (e.g. 'High')
      - probs: dict of {label: probability}
    """
    encoded = tokenizer_v2(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    model_v2.eval()
    with torch.no_grad():
        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits = outputs.logits
        probs_tensor = F.softmax(logits, dim=-1).squeeze(0)

    probs = probs_tensor.cpu().tolist()
    label_probs = {URGENCY_LABELS[i]: float(probs[i]) for i in range(len(URGENCY_LABELS))}

    predicted_idx = int(torch.argmax(probs_tensor).item())
    predicted_label = URGENCY_LABELS[predicted_idx]

    return {
        "predicted_label": predicted_label,
        "probs": label_probs,
    }


In [22]:
# Stage 2b – manual evaluation using the new model_v2

results_v2 = []

for _, row in manual_df.iterrows():
    text = row["text"]
    intended = row["intended_urgency"]

    pred = predict_urgency_v2(text)
    predicted_label = pred["predicted_label"]
    probs = pred["probs"]

    results_v2.append({
        "example_id": row["example_id"],
        "intended_urgency": intended,
        "predicted_urgency_v2": predicted_label,
        "Low_prob": probs["Low"],
        "Medium_prob": probs["Medium"],
        "High_prob": probs["High"],
        "Critical_prob": probs["Critical"],
    })

manual_eval_v2_df = pd.DataFrame(results_v2)

print("Stage 2b manual evaluation – new model vs intended labels:\n")
display(manual_eval_v2_df[["example_id", "intended_urgency", "predicted_urgency_v2"]])

# Simple accuracy on this manual set for the new model
correct_v2 = (manual_eval_v2_df["intended_urgency"] == manual_eval_v2_df["predicted_urgency_v2"]).sum()
total_v2 = len(manual_eval_v2_df)
accuracy_v2 = correct_v2 / total_v2 if total_v2 > 0 else 0.0

print(f"\nStage 2b manual set accuracy: {correct_v2} / {total_v2} = {accuracy_v2:.2f}")


Stage 2b manual evaluation – new model vs intended labels:



,example_id,intended_urgency,predicted_urgency_v2
0,1,Low,Medium
1,2,Medium,Medium
2,3,High,Low
3,4,Critical,Low
4,5,High,Medium
5,6,Medium,Low
6,7,Low,Medium
7,8,Critical,Low



Stage 2b manual set accuracy: 1 / 8 = 0.12


In [23]:
# Compare Stage 2 (old model) and Stage 2b (new model) on the manual set

comparison_rows = []

for _, row in manual_df.iterrows():
    text = row["text"]
    intended = row["intended_urgency"]

    # Stage 2 prediction (old model + tokenizer)
    pred_old = predict_urgency(text)
    old_label = pred_old["predicted_label"]
    old_probs = pred_old["probs"]

    # Stage 2b prediction (new model_v2 + tokenizer_v2)
    pred_new = predict_urgency_v2(text)
    new_label = pred_new["predicted_label"]
    new_probs = pred_new["probs"]

    comparison_rows.append({
        "example_id": row["example_id"],
        "intended_urgency": intended,
        "old_label": old_label,
        "new_label": new_label,
        "old_Low": old_probs["Low"],
        "old_Medium": old_probs["Medium"],
        "old_High": old_probs["High"],
        "old_Critical": old_probs["Critical"],
        "new_Low": new_probs["Low"],
        "new_Medium": new_probs["Medium"],
        "new_High": new_probs["High"],
        "new_Critical": new_probs["Critical"],
    })

comparison_df = pd.DataFrame(comparison_rows)

print("Stage 2 vs Stage 2b comparison on manual examples:\n")
display(comparison_df[[
    "example_id", "intended_urgency", "old_label", "new_label"
]])

print("\nFull probabilities (for deeper inspection):")
display(comparison_df)


Stage 2 vs Stage 2b comparison on manual examples:



,example_id,intended_urgency,old_label,new_label
0,1,Low,Low,Medium
1,2,Medium,Low,Medium
2,3,High,Low,Low
3,4,Critical,Medium,Low
4,5,High,High,Medium
5,6,Medium,Medium,Low
6,7,Low,Low,Medium
7,8,Critical,Medium,Low



Full probabilities (for deeper inspection):


,example_id,intended_urgency,old_label,new_label,old_Low,old_Medium,old_High,old_Critical,new_Low,new_Medium,new_High,new_Critical
0,1,Low,Low,Medium,0.324780,0.262781,0.226176,0.186263,0.059237,0.915744,0.013228,0.011791
1,2,Medium,Low,Medium,0.351643,0.269575,0.214749,0.164033,0.044226,0.921676,0.017885,0.016213
2,3,High,Low,Low,0.329533,0.294208,0.204365,0.171894,0.662133,0.264131,0.041401,0.032335
3,4,Critical,Medium,Low,0.291078,0.307420,0.236362,0.165140,0.516150,0.422746,0.031603,0.029501
4,5,High,High,Medium,0.273222,0.254314,0.277188,0.195275,0.140591,0.797876,0.035843,0.025689
5,6,Medium,Medium,Low,0.218777,0.512874,0.146242,0.122108,0.694771,0.270524,0.020429,0.014276
6,7,Low,Low,Medium,0.336339,0.244854,0.240890,0.177918,0.408534,0.542712,0.022624,0.026130
7,8,Critical,Medium,Low,0.207548,0.444737,0.181542,0.166173,0.515269,0.409127,0.040194,0.035410


In [24]:
# Show the 8 manual evaluation examples
display(manual_df[["example_id", "intended_urgency", "text"]])



,example_id,intended_urgency,text
0,1,Low,Cadet has been slightly quieter than usual and...
1,2,Medium,Cadet has started sitting on their own and avo...
2,3,High,Cadet disclosed that they have been struggling...
3,4,Critical,Cadet stated that they have a plan to harm the...
4,5,High,Cadet arrived with noticeable bruises on their...
5,6,Medium,Parent contacted the unit to say there have be...
6,7,Low,Cadet mentioned that they may need help with t...
7,8,Critical,Another cadet reported seeing threatening mess...


In [25]:
import pandas as pd

# Path to your hand-crafted 30-example manual evaluation set
manual30_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\manual\manual_urgency_examples_30.xlsx"

manual30_df = pd.read_excel(manual30_path)

print("Loaded manual 30-example dataset.")
print("Columns:", manual30_df.columns.tolist())
print(f"Number of rows: {len(manual30_df)}\n")

# If you kept my column name 'draft_urgency_suggestion',
# rename it to 'intended_urgency' for consistency.
if "draft_urgency_suggestion" in manual30_df.columns and "intended_urgency" not in manual30_df.columns:
    manual30_df = manual30_df.rename(columns={"draft_urgency_suggestion": "intended_urgency"})

# Show a preview
display(manual30_df[["example_id", "category_label", "intended_urgency", "text"]].head())


Loaded manual 30-example dataset.
Columns: ['example_id', 'category_label', 'intended_urgency', 'text']
Number of rows: 30



,example_id,category_label,intended_urgency,text
0,1,Attendance / engagement,Low,Cadet who was previously punctual has started ...
1,2,Financial hardship,Low,Cadet quietly asked if there was any support a...
2,3,Bullying,Low,Cadet reported that a small group occasionally...
3,4,Behaviour / conduct,Low,Staff have noticed the cadet has become more t...
4,5,Online safety,Low,Cadet mentioned receiving a few unwanted frien...


In [26]:
# Stage 2c – evaluate the Stage 2 model on the 30 manual examples

results_30_stage2 = []

for _, row in manual30_df.iterrows():
    text = row["text"]
    intended = row["intended_urgency"]

    # Use the original Stage 2 helper
    pred = predict_urgency(text)
    predicted_label = pred["predicted_label"]
    probs = pred["probs"]

    results_30_stage2.append({
        "example_id": row["example_id"],
        "category_label": row["category_label"],
        "intended_urgency": intended,
        "predicted_urgency": predicted_label,
        "Low_prob": probs["Low"],
        "Medium_prob": probs["Medium"],
        "High_prob": probs["High"],
        "Critical_prob": probs["Critical"],
    })

manual30_stage2_df = pd.DataFrame(results_30_stage2)

print("Stage 2 model – manual 30-example evaluation (headline view):\n")
display(manual30_stage2_df[["example_id", "category_label", "intended_urgency", "predicted_urgency"]])

# Accuracy
correct_30 = (manual30_stage2_df["intended_urgency"] == manual30_stage2_df["predicted_urgency"]).sum()
total_30 = len(manual30_stage2_df)
accuracy_30 = correct_30 / total_30 if total_30 > 0 else 0.0

print(f"\nStage 2 model accuracy on 30 manual examples: {correct_30} / {total_30} = {accuracy_30:.2f}")

# Simple cross-tab for confusion-style view
print("\nCounts by intended vs predicted urgency:\n")
crosstab = pd.crosstab(
    manual30_stage2_df["intended_urgency"],
    manual30_stage2_df["predicted_urgency"],
    rownames=["Intended"],
    colnames=["Predicted"],
    dropna=False,
)
display(crosstab)


Stage 2 model – manual 30-example evaluation (headline view):



,example_id,category_label,intended_urgency,predicted_urgency
0,1,Attendance / engagement,Low,Medium
1,2,Financial hardship,Low,Medium
2,3,Bullying,Low,Low
3,4,Behaviour / conduct,Low,Medium
4,5,Online safety,Low,Low
5,6,Mental health,Medium,Low
6,7,Home issues,Medium,Low
7,8,Bullying,Medium,Medium
8,9,Online safety,Medium,Low
9,10,Attendance / engagement,Medium,Medium



Stage 2 model accuracy on 30 manual examples: 8 / 30 = 0.27

Counts by intended vs predicted urgency:



Predicted,High,Low,Medium
Intended,,,
Critical,1,2,3
High,2,4,5
Low,0,2,3
Medium,0,4,4


In [27]:
# Stage 3 – risk-aware calibration on top of the Stage 2 model

def calibrate_urgency(label_probs: dict, raw_label: str) -> str:
    """
    Calibrate the model's raw urgency prediction using simple risk-aware rules.

    Inputs:
      - label_probs: dict like {"Low": p_low, "Medium": p_med, "High": p_high, "Critical": p_crit}
      - raw_label: model's argmax label (e.g. 'Medium')

    Returns:
      - calibrated_label: adjusted label after applying risk rules
    """

    p_low = label_probs.get("Low", 0.0)
    p_med = label_probs.get("Medium", 0.0)
    p_high = label_probs.get("High", 0.0)
    p_crit = label_probs.get("Critical", 0.0)

    # Combined upper-end risk
    high_plus_critical = p_high + p_crit

    # Rule 1: if Critical is clearly highest and reasonably large, force Critical
    if p_crit == max(p_low, p_med, p_high, p_crit) and p_crit >= 0.40:
        return "Critical"

    # Rule 2: if High + Critical probability is high, at least call it High
    if high_plus_critical >= 0.50:
        # If Critical is higher than High, treat as Critical, otherwise High
        if p_crit >= p_high:
            return "Critical"
        else:
            return "High"

    # Rule 3: if Medium is predicted but High or Critical are close behind, bump to High
    # e.g. if raw is Medium but max(High, Critical) is within 0.05 of Medium
    if raw_label == "Medium":
        upper_max = max(p_high, p_crit)
        if upper_max >= 0.30 and (p_med - upper_max) <= 0.05:
            # Conservative bump to High (rather than Critical) in borderline cases
            return "High"

    # Otherwise, fall back to the raw model label
    return raw_label


In [28]:
# Stage 3 – apply calibration to the 30 manual examples

# Make a copy so we keep the original around if needed
calibrated_30_df = manual30_stage2_df.copy()

calibrated_labels = []

for _, row in calibrated_30_df.iterrows():
    raw_label = row["predicted_urgency"]
    probs = {
        "Low": row["Low_prob"],
        "Medium": row["Medium_prob"],
        "High": row["High_prob"],
        "Critical": row["Critical_prob"],
    }

    new_label = calibrate_urgency(probs, raw_label)
    calibrated_labels.append(new_label)

calibrated_30_df["calibrated_urgency"] = calibrated_labels

print("Stage 3 – manual 30-example evaluation with calibration (headline view):\n")
display(calibrated_30_df[[
    "example_id",
    "category_label",
    "intended_urgency",
    "predicted_urgency",
    "calibrated_urgency"
]])

# Raw vs calibrated accuracy
raw_correct = (calibrated_30_df["intended_urgency"] == calibrated_30_df["predicted_urgency"]).sum()
calib_correct = (calibrated_30_df["intended_urgency"] == calibrated_30_df["calibrated_urgency"]).sum()
total = len(calibrated_30_df)

raw_acc = raw_correct / total if total > 0 else 0.0
calib_acc = calib_correct / total if total > 0 else 0.0

print(f"\nRaw model accuracy:        {raw_correct} / {total} = {raw_acc:.2f}")
print(f"Calibrated model accuracy: {calib_correct} / {total} = {calib_acc:.2f}")

print("\nRaw confusion (intended vs predicted):")
raw_crosstab = pd.crosstab(
    calibrated_30_df["intended_urgency"],
    calibrated_30_df["predicted_urgency"],
    rownames=["Intended"],
    colnames=["Predicted"],
    dropna=False,
)
display(raw_crosstab)

print("\nCalibrated confusion (intended vs calibrated):")
calib_crosstab = pd.crosstab(
    calibrated_30_df["intended_urgency"],
    calibrated_30_df["calibrated_urgency"],
    rownames=["Intended"],
    colnames=["Calibrated"],
    dropna=False,
)
display(calib_crosstab)


Stage 3 – manual 30-example evaluation with calibration (headline view):



,example_id,category_label,intended_urgency,predicted_urgency,calibrated_urgency
0,1,Attendance / engagement,Low,Medium,Medium
1,2,Financial hardship,Low,Medium,Medium
2,3,Bullying,Low,Low,Low
3,4,Behaviour / conduct,Low,Medium,Medium
4,5,Online safety,Low,Low,Low
5,6,Mental health,Medium,Low,Low
6,7,Home issues,Medium,Low,Low
7,8,Bullying,Medium,Medium,Medium
8,9,Online safety,Medium,Low,Low
9,10,Attendance / engagement,Medium,Medium,Medium



Raw model accuracy:        8 / 30 = 0.27
Calibrated model accuracy: 9 / 30 = 0.30

Raw confusion (intended vs predicted):


Predicted,High,Low,Medium
Intended,,,
Critical,1,2,3
High,2,4,5
Low,0,2,3
Medium,0,4,4



Calibrated confusion (intended vs calibrated):


Calibrated,High,Low,Medium
Intended,,,
Critical,1,2,3
High,3,3,5
Low,0,2,3
Medium,0,4,4


In [29]:
# Stage 3b – policy overrides using high-risk keywords and categories

def apply_policy_overrides(text: str, category_label: str, base_label: str) -> str:
    """
    Apply simple safeguarding policy overrides on top of the model's base urgency label.

    - text: free-text concern narrative
    - category_label: your category (e.g. 'Abuse by adult in organisation (CFAV)')
    - base_label: model's predicted label (e.g. 'Medium')

    Returns:
      final_label: urgency label after applying minimum levels based on keywords/categories.
    """

    t = text.lower().strip()

    # Helper to convert urgency to numeric for easy max() comparison
    order = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
    inv_order = {v: k for k, v in order.items()}

    current_level = order.get(base_label, 1)  # default to Medium if unknown

    # --- Keyword groups ---

    # Clear suicide / self-harm plan (Critical)
    critical_self_harm = [
        "kill myself", "end my life", "don't want to be alive",
        "no longer want to be alive", "suicide", "suicidal",
        "overdose", "take all my tablets", "hang myself",
        "jump off", "decided i no longer want to be alive",
        "plan to hurt myself", "plan for how i might harm myself",
    ]

    # Serious physical abuse (High/Critical)
    physical_abuse = [
        "hit me", "hit them", "punched", "kicked", "beat me",
        "grabbed me by the neck", "threw me", "thrown things at me",
    ]

    # Sexual abuse / CFAV abuse (High/Critical)
    sexual_abuse = [
        "touched me inappropriately", "touched them inappropriately",
        "touched me in a private area", "private area",
        "warned me not to tell", "told me not to tell anyone",
        "behind closed doors", "alone with me in the office",
    ]

    # Radicalisation / extremism
    extremism = [
        "travel to a conflict zone", "go to a conflict zone",
        "fight for an extremist cause", "join an extremist group",
        "martyr", "die for the cause",
    ]

    # Exploitation / trafficking
    exploitation = [
        "much older individuals", "much older men", "gives me money",
        "gives me gifts", "pick me up late at night",
        "locations i do not want to describe",
    ]

    # FGM / harmful practices
    fgm = [
        "special ceremony", "procedure on girls",
        "cutting ceremony", "traditional practice on girls",
    ]

    # --- Apply keyword-based minimums ---

    # Start with no policy override
    min_level = current_level

    # Suicide / self-harm with plan -> Critical
    if any(kw in t for kw in critical_self_harm):
        min_level = max(min_level, order["Critical"])

    # Physical abuse -> at least High
    if any(kw in t for kw in physical_abuse):
        min_level = max(min_level, order["High"])

    # Sexual abuse / CFAV sexualised behaviour -> at least High, often Critical
    if any(kw in t for kw in sexual_abuse):
        # If abuse by CFAV, push to Critical; otherwise High
        if "cfav" in category_label.lower() or "adult in organisation" in category_label.lower():
            min_level = max(min_level, order["Critical"])
        else:
            min_level = max(min_level, order["High"])

    # Radicalisation / extremism -> at least High
    if any(kw in t for kw in extremism):
        min_level = max(min_level, order["High"])

    # Exploitation / trafficking -> at least High
    if any(kw in t for kw in exploitation):
        min_level = max(min_level, order["High"])

    # FGM / harmful practices -> at least High
    if any(kw in t for kw in fgm):
        min_level = max(min_level, order["High"])

    # --- Category-based minimums (even without exact keywords) ---

    high_min_categories = {
        "Abuse by adult in organisation (CFAV)",
        "Exploitation / trafficking",
        "FGM / harmful practices",
        "Home issues / severe abuse",
        "Mental health / self-harm",
    }

    if category_label in high_min_categories and min_level < order["High"]:
        min_level = order["High"]

    # Convert back to label
    final_label = inv_order[min_level]
    return final_label


In [30]:
# Stage 3b – apply policy overrides to the 30 manual examples

# 1. Merge Stage 2 predictions with the original text
manual30_policy_df = manual30_stage2_df.merge(
    manual30_df[["example_id", "text"]],
    on="example_id",
    how="left",
)

# 2. Apply policy overrides
policy_labels = []

for _, row in manual30_policy_df.iterrows():
    base_label = row["predicted_urgency"]
    category_label = row["category_label"]
    text = row["text"]

    final_label = apply_policy_overrides(text=text,
                                         category_label=category_label,
                                         base_label=base_label)
    policy_labels.append(final_label)

manual30_policy_df["policy_urgency"] = policy_labels

print("Stage 3b – headline view (first 15 examples):\n")
display(manual30_policy_df[[
    "example_id",
    "category_label",
    "intended_urgency",
    "predicted_urgency",
    "policy_urgency",
]].head(15))

# 3. Accuracy: raw vs policy
raw_correct = (manual30_policy_df["intended_urgency"] == manual30_policy_df["predicted_urgency"]).sum()
policy_correct = (manual30_policy_df["intended_urgency"] == manual30_policy_df["policy_urgency"]).sum()
total = len(manual30_policy_df)

raw_acc = raw_correct / total if total > 0 else 0.0
policy_acc = policy_correct / total if total > 0 else 0.0

print(f"\nRaw model accuracy (Stage 2):     {raw_correct} / {total} = {raw_acc:.2f}")
print(f"Policy-adjusted accuracy (S3b):   {policy_correct} / {total} = {policy_acc:.2f}")

# 4. Confusion tables

print("\nRaw confusion (intended vs predicted):")
raw_crosstab = pd.crosstab(
    manual30_policy_df["intended_urgency"],
    manual30_policy_df["predicted_urgency"],
    rownames=["Intended"],
    colnames=["Predicted"],
    dropna=False,
)
display(raw_crosstab)

print("\nPolicy confusion (intended vs policy_urgency):")
policy_crosstab = pd.crosstab(
    manual30_policy_df["intended_urgency"],
    manual30_policy_df["policy_urgency"],
    rownames=["Intended"],
    colnames=["Policy"],
    dropna=False,
)
display(policy_crosstab)


Stage 3b – headline view (first 15 examples):



,example_id,category_label,intended_urgency,predicted_urgency,policy_urgency
0,1,Attendance / engagement,Low,Medium,Medium
1,2,Financial hardship,Low,Medium,Medium
2,3,Bullying,Low,Low,Low
3,4,Behaviour / conduct,Low,Medium,Medium
4,5,Online safety,Low,Low,Low
5,6,Mental health,Medium,Low,Low
6,7,Home issues,Medium,Low,Low
7,8,Bullying,Medium,Medium,Medium
8,9,Online safety,Medium,Low,Low
9,10,Attendance / engagement,Medium,Medium,Medium



Raw model accuracy (Stage 2):     8 / 30 = 0.27
Policy-adjusted accuracy (S3b):   15 / 30 = 0.50

Raw confusion (intended vs predicted):


Predicted,High,Low,Medium
Intended,,,
Critical,1,2,3
High,2,4,5
Low,0,2,3
Medium,0,4,4



Policy confusion (intended vs policy_urgency):


Policy,Critical,High,Low,Medium
Intended,,,,
Critical,2,3,1,0
High,0,7,1,3
Low,0,0,2,3
Medium,0,0,4,4


In [31]:
from pathlib import Path

# Folder to save the fine-tuned Stage 2 model and tokenizer
model_save_dir = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\urgency_distilbert_stage2_finetuned"

Path(model_save_dir).mkdir(parents=True, exist_ok=True)

print("Saving fine-tuned Stage 2 model to:")
print(model_save_dir)

# 'model' and 'tokenizer' should be the Stage 2 objects (not model_v2 / tokenizer_v2)
model.save_pretrained(model_save_dir)
tokenizer.save_pretrained(model_save_dir)

print("Save complete.")


Saving fine-tuned Stage 2 model to:
C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\urgency_distilbert_stage2_finetuned
Save complete.


In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

print("Stage 4 – DeBERTa-v3-base urgency model setup")

# Urgency labels (same as before)
URGENCY_LABELS = ["Low", "Medium", "High", "Critical"]
label2id_deberta = {label: idx for idx, label in enumerate(URGENCY_LABELS)}
id2label_deberta = {idx: label for label, idx in label2id_deberta.items()}

print("\nLabel ↔ ID mapping (DeBERTa):")
for k, v in label2id_deberta.items():
    print(f"  {k} -> {v}")

# DeBERTa model name
MODEL_NAME_DEBERTA = "microsoft/deberta-v3-base"

print(f"\nLoading tokenizer for: {MODEL_NAME_DEBERTA}")
tokenizer_deberta = AutoTokenizer.from_pretrained(MODEL_NAME_DEBERTA)

# Device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

# Check that Stage 2b splits exist
print("\nStage 2b dataframe sizes (should be 800/200 approx):")
print("  train_df_v2:", len(train_df_v2))
print("  val_df_v2:", len(val_df_v2))

# Build datasets using the existing SafeguardingUrgencyDataset class
train_dataset_deberta = SafeguardingUrgencyDataset(
    train_df_v2,
    tokenizer_deberta,
    label2id_deberta,
    max_length=128,
)
val_dataset_deberta = SafeguardingUrgencyDataset(
    val_df_v2,
    tokenizer_deberta,
    label2id_deberta,
    max_length=128,
)

print("\nStage 4 dataset sizes:")
print("  Train dataset size:", len(train_dataset_deberta))
print("  Validation dataset size:", len(val_dataset_deberta))

# DataLoaders – small batch size to be safe on VRAM
BATCH_SIZE_DEBERTA = 8

train_dataloader_deberta = DataLoader(
    train_dataset_deberta,
    batch_size=BATCH_SIZE_DEBERTA,
    shuffle=True,
)
val_dataloader_deberta = DataLoader(
    val_dataset_deberta,
    batch_size=BATCH_SIZE_DEBERTA,
    shuffle=False,
)

print("\nStage 4 DataLoader batches:")
print("  Train batches:", len(train_dataloader_deberta))
print("  Validation batches:", len(val_dataloader_deberta))

# Load DeBERTa-v3-base for sequence classification
print(f"\nLoading DeBERTa-v3-base model for 4-way classification...")
model_deberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DEBERTA,
    num_labels=len(URGENCY_LABELS),
    id2label=id2label_deberta,
    label2id=label2id_deberta,
)

model_deberta.to(device)

print("\nStage 4 – DeBERTa model loaded and moved to device.")


Stage 4 – DeBERTa-v3-base urgency model setup

Label ↔ ID mapping (DeBERTa):
  Low -> 0
  Medium -> 1
  High -> 2
  Critical -> 3

Loading tokenizer for: microsoft/deberta-v3-base


c:\Users\Alex\Documents\dissertation-ai\.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



Using device: cuda

Stage 2b dataframe sizes (should be 800/200 approx):
  train_df_v2: 800
  val_df_v2: 200

Stage 4 dataset sizes:
  Train dataset size: 800
  Validation dataset size: 200

Stage 4 DataLoader batches:
  Train batches: 100
  Validation batches: 25

Loading DeBERTa-v3-base model for 4-way classification...


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Stage 4 – DeBERTa model loaded and moved to device.


In [33]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

print("Stage 4 – Training DeBERTa-v3-base on Stage 2b synthetic data")

# Training config – we can tune later if needed
EPOCHS_DEBERTA = 3
LEARNING_RATE_DEBERTA = 2e-5

optimizer_deberta = torch.optim.AdamW(model_deberta.parameters(), lr=LEARNING_RATE_DEBERTA)

total_steps_deberta = len(train_dataloader_deberta) * EPOCHS_DEBERTA

scheduler_deberta = get_linear_schedule_with_warmup(
    optimizer_deberta,
    num_warmup_steps=0,
    num_training_steps=total_steps_deberta,
)

print(f"DeBERTa training for {EPOCHS_DEBERTA} epochs with learning rate {LEARNING_RATE_DEBERTA}.")


def evaluate_model_deberta(model, dataloader, device):
    """
    Simple accuracy evaluation for the DeBERTa urgency model.
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total if total > 0 else 0.0
    return accuracy


# Main training loop for Stage 4 (DeBERTa)
for epoch in range(EPOCHS_DEBERTA):
    model_deberta.train()
    running_loss = 0.0

    print(f"\n=== Stage 4 – Epoch {epoch + 1} / {EPOCHS_DEBERTA} ===")

    for step, batch in enumerate(train_dataloader_deberta):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_deberta.zero_grad()

        outputs = model_deberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model_deberta.parameters(), max_norm=1.0)

        optimizer_deberta.step()
        scheduler_deberta.step()

        running_loss += loss.item()

        if (step + 1) % 20 == 0 or (step + 1) == len(train_dataloader_deberta):
            avg_loss_so_far = running_loss / (step + 1)
            print(f"  Step {step + 1}/{len(train_dataloader_deberta)} - "
                  f"Avg train loss: {avg_loss_so_far:.4f}")

    # End of epoch – evaluate on validation set
    val_accuracy_deberta = evaluate_model_deberta(model_deberta, val_dataloader_deberta, device)
    avg_train_loss_deberta = running_loss / len(train_dataloader_deberta)

    print(f"\nStage 4 – Epoch {epoch + 1} summary:")
    print(f"  Average training loss: {avg_train_loss_deberta:.4f}")
    print(f"  Validation accuracy:   {val_accuracy_deberta:.4f}")


Stage 4 – Training DeBERTa-v3-base on Stage 2b synthetic data
DeBERTa training for 3 epochs with learning rate 2e-05.

=== Stage 4 – Epoch 1 / 3 ===
  Step 20/100 - Avg train loss: 1.3450
  Step 40/100 - Avg train loss: 1.2410
  Step 60/100 - Avg train loss: 1.1380
  Step 80/100 - Avg train loss: 0.9839
  Step 100/100 - Avg train loss: 0.8252

Stage 4 – Epoch 1 summary:
  Average training loss: 0.8252
  Validation accuracy:   1.0000

=== Stage 4 – Epoch 2 / 3 ===
  Step 20/100 - Avg train loss: 0.0464
  Step 40/100 - Avg train loss: 0.0310
  Step 60/100 - Avg train loss: 0.0238
  Step 80/100 - Avg train loss: 0.0197
  Step 100/100 - Avg train loss: 0.0170

Stage 4 – Epoch 2 summary:
  Average training loss: 0.0170
  Validation accuracy:   1.0000

=== Stage 4 – Epoch 3 / 3 ===
  Step 20/100 - Avg train loss: 0.0057
  Step 40/100 - Avg train loss: 0.0052
  Step 60/100 - Avg train loss: 0.0050
  Step 80/100 - Avg train loss: 0.0047
  Step 100/100 - Avg train loss: 0.0045

Stage 4 – Epoch 

In [34]:
import torch.nn.functional as F

print("Stage 4 – DeBERTa prediction helper on manual 30 examples")

def predict_urgency_deberta(text: str) -> dict:
    """
    Use the Stage 4 DeBERTa-v3-base model to predict urgency for a single text.

    Returns:
      - predicted_label: string urgency label (e.g. 'High')
      - probs: dict of {label: probability}
    """
    encoded = tokenizer_deberta(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    model_deberta.eval()
    with torch.no_grad():
        outputs = model_deberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits = outputs.logits
        probs_tensor = F.softmax(logits, dim=-1).squeeze(0)

    probs = probs_tensor.cpu().tolist()
    label_probs = {URGENCY_LABELS[i]: float(probs[i]) for i in range(len(URGENCY_LABELS))}

    predicted_idx = int(torch.argmax(probs_tensor).item())
    predicted_label = URGENCY_LABELS[predicted_idx]

    return {
        "predicted_label": predicted_label,
        "probs": label_probs,
    }


Stage 4 – DeBERTa prediction helper on manual 30 examples


In [ ]:
# Stage 4 – evaluate DeBERTa on the 30 manual examples

results_30_deberta = []

for _, row in manual30_df.iterrows():
    text = row["text"]
    intended = row["intended_urgency"]

    pred = predict_urgency_deberta(text)
    predicted_label = pred["predicted_label"]
    probs = pred["probs"]

    results_30_deberta.append({
        "example_id": row["example_id"],
        "category_label": row["category_label"],
        "intended_urgency": intended,
        "predicted_urgency_deberta": predicted_label,
        "Low_prob": probs["Low"],
        "Medium_prob": probs["Medium"],
        "High_prob": probs["High"],
        "Critical_prob": probs["Critical"],
    })

manual30_deberta_df = pd.DataFrame(results_30_deberta)

print("Stage 4 – DeBERTa model on 30 manual examples (headline view):\n")
display(manual30_deberta_df[[
    "example_id", "category_label", "intended_urgency", "predicted_urgency_deberta"
]])

# Accuracy
correct_30_deberta = (manual30_deberta_df["intended_urgency"] == manual30_deberta_df["predicted_urgency_deberta"]).sum()
total_30 = len(manual30_deberta_df)
accuracy_30_deberta = correct_30_deberta / total_30 if total_30 > 0 else 0.0

print(f"\nStage 4 – DeBERTa accuracy on 30 manual examples: {correct_30_deberta} / {total_30} = {accuracy_30_deberta:.2f}")

print("\nCounts by intended vs DeBERTa-predicted urgency:\n")
crosstab_deberta = pd.crosstab(
    manual30_deberta_df["intended_urgency"],
    manual30_deberta_df["predicted_urgency_deberta"],
    rownames=["Intended"],
    colnames=["Predicted (DeBERTa)"],
    dropna=False,
)
display(crosstab_deberta)


Stage 4 – DeBERTa model on 30 manual examples (headline view):



,example_id,category_label,intended_urgency,predicted_urgency_deberta
0,1,Attendance / engagement,Low,Medium
1,2,Financial hardship,Low,Medium
2,3,Bullying,Low,Low
3,4,Behaviour / conduct,Low,Low
4,5,Online safety,Low,Medium
5,6,Mental health,Medium,Medium
6,7,Home issues,Medium,Medium
7,8,Bullying,Medium,Medium
8,9,Online safety,Medium,Medium
9,10,Attendance / engagement,Medium,Low



Stage 4 – DeBERTa accuracy on 30 manual examples: 9 / 30 = 0.30

Counts by intended vs DeBERTa-predicted urgency:



Predicted (DeBERTa),Low,Medium
Intended,,
Critical,0,6
High,0,11
Low,2,3
Medium,1,7


: 